In [ ]:
import pandas as pd

# 파일 경로
input_file = "simulation_results.csv"

# 데이터 로드
df = pd.read_csv(input_file)

# 0 데이터 제거 (입력 전체가 0이거나, 주요 입력 컬럼들의 합계가 0인 행 제거)
input_cols = df.columns[:df.columns.get_loc('sl2_side_y')+1]
df = df[df[input_cols].sum(axis=1) != 0]

# Outlier 필터링 함수 (IQR 방식)
def remove_outliers(df, cols, weight=1.5):
    filtered_df = df.copy()
    for col in cols:
        Q1 = filtered_df[col].quantile(0.25)
        Q3 = filtered_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - weight * IQR
        upper = Q3 + weight * IQR
        filtered_df = filtered_df[(filtered_df[col] >= lower) & (filtered_df[col] <= upper)]
    return filtered_df

# 전체 입력 + 타겟 추가 컬럼 목록 정의
target_cols = [
    'Ltx','Lrx','M','k',
    'Lmt','Lmr','Llt','Llr',
    'Tx_loss','Rx_loss',
    'P_Tx_main_winding_inner','P_Tx_main_winding_outer',
    'P_Tx_side_winding_inner','P_Tx_side_winding_outer',
    'time'
]



for target in target_cols:
    save_cols = list(input_cols) + [target]
    # 해당 컬럼만 선택
    data = df[save_cols].copy()
    # 0 데이터 제거 (추가: 타겟 값이 0 인 행도 제거)
    # 1e-6보다 작은 값(절대값 기준)도 0처럼 간주하여 제거
    data = data[~(((data[input_cols].sum(axis=1) != 0) & (data[input_cols].sum(axis=1).abs() < 1e-6)) | ((data[target] != 0) & (data[target].abs() < 1e-6)))]
    # 입력+타겟에 대해 이상치 제거
    data_clean = remove_outliers(data, cols=list(input_cols) + [target])
    out_file = f"data_{target}.csv"
    data_clean.to_csv(out_file, index=False)
    print(f"Saved: {out_file}, shape = {data_clean.shape}")

Saved: data_Ltx.csv, shape = (21599, 50)
Saved: data_Lrx.csv, shape = (21604, 50)
Saved: data_M.csv, shape = (21601, 50)
Saved: data_k.csv, shape = (20986, 50)
Saved: data_Lmt.csv, shape = (21599, 50)
Saved: data_Lmr.csv, shape = (21597, 50)
Saved: data_Llt.csv, shape = (20994, 50)
Saved: data_Llr.csv, shape = (20995, 50)
Saved: data_Tx_loss.csv, shape = (21085, 50)
Saved: data_Rx_loss.csv, shape = (21493, 50)
Saved: data_P_Tx_main_winding_inner.csv, shape = (21314, 50)
Saved: data_P_Tx_main_winding_outer.csv, shape = (20496, 50)
Saved: data_P_Tx_side_winding_inner.csv, shape = (20168, 50)
Saved: data_P_Tx_side_winding_outer.csv, shape = (21396, 50)
Saved: data_time.csv, shape = (21352, 50)
